# Как получить отчёт Evidently
Сравним две среды в данных жёлтых такси Нью-Йорка и проверим, заметит ли Evidently надбавку к тарифу. Для запуска нужен Parquet за ноябрь 2023 года. [Подготовка данных](README.md#4-как-повторить-локально).

In [1]:
from pathlib import Path
import os

from evidently import Report
from evidently.presets import DataDriftPreset

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "seminar/evidently/quickstart/drift.py").is_file())
os.chdir(root)
from seminar.evidently.quickstart.drift import OUTPUT, trips_on

## 1. Собрать reference и current
Берём поездки по местной дате посадки. Функция применяет одинаковые фильтры и возвращает пять признаков. [Код чтения Parquet](drift.py#L13).

In [2]:
reference = trips_on("2023-11-08")
current = trips_on("2023-11-15")
print(current.columns.tolist())

2023-11-08 117271 trips; median fare: 14.2
payment_type
card    0.859
cash    0.141
Name: proportion, dtype: float64
2023-11-15 117344 trips; median fare: 14.2
payment_type
card    0.8591
cash    0.1409
Name: proportion, dtype: float64
['fare_amount', 'payment_type', 'trip_distance', 'passenger_count', 'PULocationID']


## 2. Выбрать метод для каждого типа данных
Числа сравниваем нормированным расстоянием Вассерштейна, категории — расстоянием Дженсена — Шеннона. Для этой демонстрации используем порог 0,1 у обоих методов.

In [3]:
report = Report([DataDriftPreset(
    num_method="wasserstein", num_threshold=0.1,
    cat_method="jensenshannon", cat_threshold=0.1,
)])

## 3. Получить отчёт до вмешательства
`.run()` рассчитывает метрики. `.save_html()` сохраняет интерактивный отчёт, который можно открыть в браузере.

In [4]:
control = report.run(reference_data=reference, current_data=current)
control.save_html(str(OUTPUT / "control.html"))

## 4. Изменить один признак и повторить проверку
Оставляем те же строки current. В первой копии увеличиваем каждый тариф на 10 долларов. Во второй копии моделируем ошибку записи способа оплаты. Reference общий для всех запусков.

In [5]:
shifted = current.assign(fare_amount=current["fare_amount"] + 10)
result = report.run(reference_data=reference, current_data=shifted)
result.save_html(str(OUTPUT / "shifted.html"))

In [6]:
payment = current.assign(payment_type="cash")
result = report.run(reference_data=reference, current_data=payment)
result.save_html(str(OUTPUT / "payment.html"))

**Вывод.** При надбавке срабатывает проверка тарифа (0,6075), а при ошибке маппинга — способа оплаты (0,6927). В обоих случаях порог пересёк один признак из пяти. [Методы и пороги Evidently](https://docs.evidentlyai.com/metrics/customize_data_drift).